# Solution — Task 02: Aggregations and Joins

## Setup

In [ ]:
import os, shutil, subprocess, sys

def _find_java():
    """Check if java is available on PATH or in JAVA_HOME."""
    if os.environ.get("JAVA_HOME"):
        java_bin = os.path.join(os.environ["JAVA_HOME"], "bin", "java")
        if os.path.isfile(java_bin):
            return True
    return shutil.which("java") is not None

def _find_installed_jdk():
    """Look for a previously installed JDK in ~/.jdk."""
    jdk_dir = os.path.expanduser("~/.jdk")
    if os.path.exists(jdk_dir):
        for d in sorted(os.listdir(jdk_dir)):
            candidate = os.path.join(jdk_dir, d)
            if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "bin", "java")):
                return candidate
    return None

# Auto-install Java if not available (required by PySpark)
if not _find_java():
    prev = _find_installed_jdk()
    if prev:
        os.environ["JAVA_HOME"] = prev
        print(f"Using JAVA_HOME={prev}")
    else:
        print("Java not found. Installing JDK 17 via install-jdk...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "install-jdk"])
        import jdk
        path = jdk.install("17")
        os.environ["JAVA_HOME"] = path
        print(f"JAVA_HOME set to {path}")
else:
    print(f"Java found. JAVA_HOME={os.environ.get('JAVA_HOME', '(system)')}")

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Task02") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

emp_df = spark.read.csv(os.path.join(FIXTURES, "employees.csv"), header=True, inferSchema=True)
orders_df = spark.read.csv(os.path.join(FIXTURES, "orders.csv"), header=True, inferSchema=True)
customers_df = spark.read.csv(os.path.join(FIXTURES, "customers.csv"), header=True, inferSchema=True)
print("Setup complete.")

## Solution 2.1: Department statistics

In [ ]:
dept_stats = emp_df.groupBy("department").agg(
    F.count("*").alias("num_employees"),
    F.round(F.avg("salary"), 0).alias("avg_salary"),
    F.max("salary").alias("max_salary"),
).orderBy(F.col("avg_salary").desc())
dept_stats.show()

# TEST — Do not modify
rows = {r["department"]: r for r in dept_stats.collect()}
assert len(rows) == 3
assert rows["Engineering"]["num_employees"] == 6
assert rows["Engineering"]["max_salary"] == 105000
collected = dept_stats.collect()
avg_salaries = [r["avg_salary"] for r in collected]
assert avg_salaries == sorted(avg_salaries, reverse=True), "Should be sorted by avg_salary desc"
print("Task 2.1 passed!")

## Solution 2.2: Join orders with customers

In [ ]:
order_details = orders_df.join(customers_df, on="customer_id", how="inner") \
    .select("order_id", "name", "product", "amount", "city") \
    .orderBy("order_id")
order_details.show()

# TEST — Do not modify
assert order_details.count() == 15
assert set(order_details.columns) == {"order_id", "name", "product", "amount", "city"}
first = order_details.collect()[0]
assert first["order_id"] == 101
assert first["name"] == "Alice"
assert first["city"] == "New York"
print("Task 2.2 passed!")

## Solution 2.3: Customer spending summary

In [ ]:
customer_spending = orders_df.join(customers_df, on="customer_id") \
    .groupBy("customer_id", "name", "city") \
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("amount").alias("total_spent"),
    ) \
    .orderBy(F.col("total_spent").desc())
customer_spending.show()

# TEST — Do not modify
assert customer_spending.count() == 5
rows = {r["customer_id"]: r for r in customer_spending.collect()}
assert rows[1]["total_orders"] == 4
assert rows[1]["total_spent"] == 2750.0
spending_vals = [r["total_spent"] for r in customer_spending.collect()]
assert spending_vals == sorted(spending_vals, reverse=True)
print("Task 2.3 passed!")

## Solution 2.4: Pivot — spending by product per customer

In [ ]:
pivot_df = orders_df.groupBy("customer_id") \
    .pivot("product") \
    .agg(F.sum("amount")) \
    .fillna(0) \
    .orderBy("customer_id")
pivot_df.show()

# TEST — Do not modify
rows = {r["customer_id"]: r for r in pivot_df.collect()}
assert rows[1]["Laptop"] == 1200.0
assert rows[1]["Headphones"] == 150.0
assert rows[4]["Laptop"] == 0  # or 0.0
assert pivot_df.count() == 5
print("Task 2.4 passed!")

## Solution 2.5: Product revenue ranking

In [ ]:
from pyspark.sql import Window

product_agg = orders_df.groupBy("product").agg(
    F.sum("amount").alias("total_revenue"),
    F.count("*").alias("order_count"),
)

w = Window.orderBy(F.col("total_revenue").desc())
product_ranking = product_agg.withColumn("revenue_rank", F.row_number().over(w))
product_ranking.show()

# TEST — Do not modify
rows = {r["product"]: r for r in product_ranking.collect()}
assert rows["Laptop"]["revenue_rank"] == 1  # highest revenue
assert rows["Laptop"]["total_revenue"] == 4950.0
assert rows["Laptop"]["order_count"] == 4
print("Task 2.5 passed!")

## Cleanup

In [ ]:
spark.stop()
print("All tasks done!")